In [ ]:
var file = @".\*.pdf";
// var file = @".\sample.jpg";

In [ ]:
#r "nuget:Azure.Identity"
#r "nuget:Azure.AI.DocumentIntelligence"

In [ ]:
using Azure.Identity;
using Azure.AI.DocumentIntelligence;
using Azure;
using System.IO;

string endpoint = "https://e.services.ai.azure.com/";
var credential = new DefaultAzureCredential(new DefaultAzureCredentialOptions
{
    TenantId = ""
});
var client = new DocumentIntelligenceClient(new Uri(endpoint), credential);

In [ ]:
var bianryData = File.ReadAllBytes(file);
Operation<AnalyzeResult> operation = await client.AnalyzeDocumentAsync(WaitUntil.Completed,  "prebuilt-invoice", BinaryData.FromBytes(bianryData));
AnalyzeResult result = operation.Value;

In [ ]:
foreach (DocumentPage page in result.Pages)
{
    Console.WriteLine($"Document Page {page.PageNumber} has {page.Lines.Count} line(s), {page.Words.Count} word(s)," +
        $" and {page.SelectionMarks.Count} selection mark(s).");

    for (int i = 0; i < page.Lines.Count; i++)
    {
        DocumentLine line = page.Lines[i];

        Console.WriteLine($"  Line {i}:");
        Console.WriteLine($"    Content: '{line.Content}'");

        Console.Write("    Bounding polygon, with points ordered clockwise:");
        for (int j = 0; j < line.Polygon.Count; j += 2)
        {
            Console.Write($" ({line.Polygon[j]}, {line.Polygon[j + 1]})");
        }

        Console.WriteLine();
    }

    for (int i = 0; i < page.SelectionMarks.Count; i++)
    {
        DocumentSelectionMark selectionMark = page.SelectionMarks[i];

        Console.WriteLine($"  Selection Mark {i} is {selectionMark.State}.");
        Console.WriteLine($"    State: {selectionMark.State}");

        Console.Write("    Bounding polygon, with points ordered clockwise:");
        for (int j = 0; j < selectionMark.Polygon.Count; j++)
        {
            Console.Write($" ({selectionMark.Polygon[j]}, {selectionMark.Polygon[j + 1]})");
        }

        Console.WriteLine();
    }
}

In [ ]:
for (int i = 0; i < result.Paragraphs.Count; i++)
{
    DocumentParagraph paragraph = result.Paragraphs[i];

    Console.WriteLine($"Paragraph {i}:");
    Console.WriteLine($"  Content: {paragraph.Content}");

    if (paragraph.Role != null)
    {
        Console.WriteLine($"  Role: {paragraph.Role}");
    }
}


In [ ]:
foreach (DocumentStyle style in result.Styles)
{
    // Check the style and style confidence to see if text is handwritten.
    // Note that value '0.8' is used as an example.

    bool isHandwritten = style.IsHandwritten.HasValue && style.IsHandwritten == true;

    if (isHandwritten && style.Confidence > 0.8)
    {
        Console.WriteLine($"Handwritten content found:");

        foreach (DocumentSpan span in style.Spans)
        {
            var handwrittenOptions = result.Content.Substring(span.Offset, span.Length);
            Console.WriteLine($"  {handwrittenOptions}");
        }
    }
}


In [ ]:
for (int i = 0; i < result.Tables.Count; i++)
{
    DocumentTable table = result.Tables[i];

    Console.WriteLine($"Table {i} has {table.RowCount} rows and {table.ColumnCount} columns.");

    foreach (DocumentTableCell cell in table.Cells)
    {
        Console.WriteLine($"  Cell ({cell.RowIndex}, {cell.ColumnIndex}) is a '{cell.Kind}' with content: {cell.Content}");
    }
}

In [ ]:
using System.Text.Json;
    var options = new JsonSerializerOptions { WriteIndented = true };
JsonSerializer.Serialize(result.Tables, options)

In [ ]:
// =============================================================
// Transform raw Document Intelligence tables into a generic
// normalized schema that is easy to process downstream.
// =============================================================
using System.Text.Json;
using System.Text.Json.Nodes;
using System.Linq;

JsonArray BuildGenericTables(IReadOnlyList<DocumentTable> tables)
{
    var output = new JsonArray();

    for (int t = 0; t < tables.Count; t++)
    {
        var table = tables[t];
        var cells = table.Cells;

        // Detect header row: row 0 cells with Kind == "columnHeader"
        bool hasHeaderRow = cells.Any(c => c.RowIndex == 0
            && c.Kind != null
            && c.Kind.ToString() == "columnHeader");

        // Heuristic: a 2-column table where every row is label→value
        // is key-value even if row 0 is marked as header
        bool isKeyValue = table.ColumnCount == 2;

        // Build header map (column index → header name) with span dedup
        var headers = new Dictionary<int, string>();
        if (hasHeaderRow && !isKeyValue)
        {
            foreach (var cell in cells.Where(c => c.RowIndex == 0).OrderBy(c => c.ColumnIndex))
            {
                int span = cell.ColumnSpan ?? 1;
                for (int cs = 0; cs < span; cs++)
                {
                    string name = cell.Content;
                    // Disambiguate duplicate names from ColumnSpan
                    if (span > 1 && cs > 0)
                        name = $"{cell.Content}_{cs + 1}";
                    headers[cell.ColumnIndex + cs] = name;
                }
            }
        }

        var tableObj = new JsonObject
        {
            ["tableIndex"] = t,
            ["rowCount"] = table.RowCount,
            ["columnCount"] = table.ColumnCount,
            ["type"] = isKeyValue ? "key-value" : "columnar",
            ["page"] = table.BoundingRegions.Count > 0 ? table.BoundingRegions[0].PageNumber : 0
        };

        if (isKeyValue)
        {
            // --- Key-Value table: col 0 = label, col 1 = value for ALL rows ---
            var kvPairs = new JsonObject();
            foreach (var cell in cells.Where(c => c.ColumnIndex == 0).OrderBy(c => c.RowIndex))
            {
                string key = cell.Content?.Trim();
                if (string.IsNullOrEmpty(key)) key = $"row_{cell.RowIndex}";
                string value = cells
                    .FirstOrDefault(c => c.RowIndex == cell.RowIndex && c.ColumnIndex == 1)?
                    .Content?.Trim() ?? "";
                kvPairs[key] = value;
            }
            tableObj["data"] = kvPairs;
        }
        else
        {
            // --- Columnar table ---
            if (headers.Count > 0)
            {
                var headersArray = new JsonArray();
                foreach (var h in headers.OrderBy(kv => kv.Key))
                    headersArray.Add(h.Value);
                tableObj["headers"] = headersArray;
            }

            int dataStartRow = hasHeaderRow ? 1 : 0;
            var rows = new JsonArray();

            for (int r = dataStartRow; r < table.RowCount; r++)
            {
                var rowCells = cells.Where(c => c.RowIndex == r).OrderBy(c => c.ColumnIndex).ToList();

                // Is this a spanning section-header row?
                bool isSpanningRow = rowCells.Any(c => (c.ColumnSpan ?? 1) > table.ColumnCount / 2);

                if (isSpanningRow && rowCells.Count <= 2)
                {
                    var spanContent = string.Join(" ",
                        rowCells.Select(c => c.Content?.Trim()).Where(s => !string.IsNullOrEmpty(s)));
                    if (!string.IsNullOrEmpty(spanContent))
                    {
                        rows.Add(new JsonObject
                        {
                            ["_type"] = "section-header",
                            ["content"] = spanContent
                        });
                    }
                }
                else
                {
                    var rowObj = new JsonObject();
                    bool allEmpty = true;

                    foreach (var cell in rowCells)
                    {
                        string content = cell.Content?.Trim() ?? "";
                        if (!string.IsNullOrEmpty(content)) allEmpty = false;

                        string colName = headers.ContainsKey(cell.ColumnIndex)
                            ? headers[cell.ColumnIndex]
                            : $"col_{cell.ColumnIndex}";
                        rowObj[colName] = content;
                    }

                    if (!allEmpty)
                        rows.Add(rowObj);
                }
            }
            tableObj["rows"] = rows;
        }

        output.Add(tableObj);
    }

    return output;
}

var genericSchema = BuildGenericTables(result.Tables);
// genericSchema.Display();


var prettyOptions = new JsonSerializerOptions
{
    WriteIndented = true,
    Encoder = System.Text.Encodings.Web.JavaScriptEncoder.UnsafeRelaxedJsonEscaping
};
var json = genericSchema.ToJsonString(prettyOptions);
Console.WriteLine(json);